### Triplet Contrastive Loss
https://arxiv.org/pdf/1503.03832.pdf

Simple triplet margin loss implementation as a baseline. 
- Anchor with single positive and negative.
- Positive is most similar 
- Negative is least similar

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from stock_embeddings import ReturnsData
PERIOD = 20
data = ReturnsData()
data.change_returns_period(PERIOD)
X = data.returns_df.select(data.tickers).to_numpy().T

In [3]:
X.shape

(611, 234)

In [4]:
num_TS = X.shape[0]
num_pos_samples = 1
period = 10
stride = 3
num_neg_samples = 1

print(f"Context Size: {num_pos_samples}, Period: {period}, Stride: {stride}")
print(f"Number of Negative Samples: {num_neg_samples}")

from stock_embeddings.context.pointwise import get_tgt_context_euclidean_multiprocess
positive_tgt_context_sets = get_tgt_context_euclidean_multiprocess(ts_array=X, m=period, k=num_pos_samples, stride=stride, z_normalize=False, verbose=False)
negative_tgt_context_sets = get_tgt_context_euclidean_multiprocess(ts_array=X, m=period, k=num_neg_samples, stride=stride, z_normalize=False, top_k=False, verbose=False)
print(f"Number (anchor, positive, negative) samples: {len(positive_tgt_context_sets)}")

Context Size: 1, Period: 10, Stride: 3
Number of Negative Samples: 1


Number (anchor, positive, negative) samples: 45825


In [5]:
index_samples = []
for pos, neg in zip(positive_tgt_context_sets,negative_tgt_context_sets):
    index_samples.append((pos[0], pos[1][0], neg[1][0]))

In [6]:
import torch
def normalize_embeddings(embeddings):
    with torch.no_grad():  # Ensure no gradients are computed in this block
        norms = embeddings.weight.norm(dim=1, keepdim=True)
        embeddings.weight.data = embeddings.weight.data / norms
    return embeddings

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Parameters
num_items = num_TS  # Number of items in your dataset
embedding_size = 16  # Size of each embedding
learning_rate = 0.001
epochs = 100
batch_size = 2048

# Initialize the embedding matrix
embeddings = nn.Embedding(num_embeddings=num_items, embedding_dim=embedding_size)

# Triplet margin loss
triplet_loss = nn.TripletMarginLoss(margin=1.0, p=2)

# Optimizer
optimizer = optim.Adam(embeddings.parameters(), lr=learning_rate)

# Prepare your index_samples as a tensor
# index_samples = torch.tensor([(35, 12, 98), (47, 12, 4), ...])  # Your index tuples as a tensor
# DataLoader for batching
dataset = TensorDataset(torch.tensor(index_samples))
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Training loop with batching
for epoch in range(epochs):
    total_loss = 0
    embeddings = normalize_embeddings(embeddings)
    for batch in data_loader:
        # Get the embeddings for anchor, positive, and negative
        batch_embeddings = embeddings(batch[0]) # shape: (batch_size, 3, embedding_dim)
        anchor_embedding = batch_embeddings[:,0,:]
        positive_embedding = batch_embeddings[:,1,:]
        negative_embedding = batch_embeddings[:,2,:]

        # Compute the loss
        loss = triplet_loss(anchor_embedding, positive_embedding, negative_embedding)
        total_loss += loss.item()

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch % 10 == 0) | (epoch==epochs-1):
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(data_loader)}')

# Example index_samples
# index_samples = [(35, 12, 98), (47, 12, 4), ...]  # Your index tuples


Epoch [1/100], Loss: 0.9568968156109685


Epoch [11/100], Loss: 0.8699122485907181


Epoch [21/100], Loss: 0.6154337214386981


Epoch [31/100], Loss: 0.19568233321542325


Epoch [41/100], Loss: 0.12156748868848967


...

Epoch [61/100], Loss: 0.10099749169919801


Epoch [71/100], Loss: 0.09588061629430107


Epoch [81/100], Loss: 0.09042193643424822


Epoch [91/100], Loss: 0.08683920100979182


Epoch [100/100], Loss: 0.08389576589283736


### Get training data

In [8]:
from stock_embeddings import get_sector_score

get_sector_score(embeddings.weight.detach().numpy(), sectors=data.sectors, top_k_accuracy=True)

/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=Tru

Precision Score: 0.25
Recall Score: 0.24
F1 Score: 0.21
Accuracy Score: 0.24
Accuracy Score Top-3: 0.55


/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

In [9]:
from stock_embeddings.plotting import pca_plot_from_embeddings

pca_plot_from_embeddings(
    embedding_matrix=embeddings.weight.detach().numpy(),
    sectors=data.sectors,
    tickers=data.tickers,
    industries=data.industries,
    names=data.names,
    dimensions=2,
    reduced=True,
    method="PCA",
    return_df=False,
    rand_state=None,
)